In [15]:
import os
import numpy as np
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# TensorFlow imports
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking, TimeDistributed
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [16]:
np.random.seed(42)
tf.random.set_seed(42)

In [17]:
def load_labeled_data(labeled_behaviors_dir):
    """Load all labeled person data from the directory"""
    all_sequences = []
    all_labels = []
    all_metadata = []

    # Find all npz files
    npz_files = [f for f in os.listdir(labeled_behaviors_dir) if f.endswith('_data.npz')]

    print(f"Found {len(npz_files)} labeled person files")

    for npz_file in tqdm(npz_files, desc="Loading data"):
        # Load the data
        data = np.load(os.path.join(labeled_behaviors_dir, npz_file))
        skeletons = data['skeletons']  # (n_frames, 34)
        behaviors = data['behaviors']   # (n_frames,)

        # Skip very short sequences
        if len(skeletons) < 10:
            continue

        # Check for corresponding JSON file for metadata
        json_file = npz_file.replace('_data.npz', '_tracking.json')
        json_path = os.path.join(labeled_behaviors_dir, json_file)

        metadata = {}
        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                metadata = json.load(f)

        all_sequences.append(skeletons)
        all_labels.append(behaviors)
        all_metadata.append(metadata)

    return all_sequences, all_labels, all_metadata

In [18]:
# Load the data
LABELED_DIR = "labeled_behaviors"
sequences, labels, metadata = load_labeled_data(LABELED_DIR)

print(f"\nLoaded {len(sequences)} person sequences")
print(f"Sequence lengths: min={min(len(s) for s in sequences)}, "
      f"max={max(len(s) for s in sequences)}, "
      f"mean={np.mean([len(s) for s in sequences]):.1f}")

Found 4 labeled person files


Loading data: 100%|██████████| 4/4 [00:00<00:00, 41.24it/s]


Loaded 4 person sequences
Sequence lengths: min=46, max=58, mean=49.0


In [19]:
# Count behavior distributions
behavior_counts = {0: 0, 1: 0, 2: 0}
for label_seq in labels:
    for label in label_seq:
        if label in behavior_counts:
            behavior_counts[label] += 1

print("\nBehavior distribution across all frames:")
total_frames = sum(behavior_counts.values())
for behavior, count in behavior_counts.items():
    behavior_names = {0: "walking/nothing", 1: "suspicious", 2: "running/panic"}
    print(f"{behavior_names[behavior]}: {count} frames ({100*count/total_frames:.1f}%)")


Behavior distribution across all frames:
walking/nothing: 188 frames (95.9%)
suspicious: 0 frames (0.0%)
running/panic: 8 frames (4.1%)


In [20]:
def create_fixed_length_sequences(sequences, labels, window_size=30, step=10):
    """Create fixed-length sequences using sliding window"""
    X_sequences = []
    y_sequences = []

    for seq, lab in zip(sequences, labels):
        # Skip sequences shorter than window
        if len(seq) < window_size:
            continue

        # Sliding window
        for i in range(0, len(seq) - window_size + 1, step):
            X_sequences.append(seq[i:i + window_size])
            y_sequences.append(lab[i:i + window_size])

    return np.array(X_sequences), np.array(y_sequences)

In [21]:
WINDOW_SIZE = 30  # 30 frames = 30 seconds at 1 FPS
STEP_SIZE = 10    # Overlap of 20 frames

X, y = create_fixed_length_sequences(sequences, labels, WINDOW_SIZE, STEP_SIZE)
print(f"\nCreated {len(X)} fixed-length sequences")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# ## 4. Handle Missing Data (NaN values)

# Replace NaN values with 0 and create mask
mask = ~np.isnan(X)
X = np.nan_to_num(X, nan=0.0)


Created 9 fixed-length sequences
X shape: (9, 30, 34)
y shape: (9, 30)


In [22]:
non_zero_mask = X != 0
if non_zero_mask.any():
    global_min = X[non_zero_mask].min()
    global_max = X[non_zero_mask].max()
    X = (X - global_min) / (global_max - global_min + 1e-8)
    X[~mask] = 0  # Re-zero the NaN positions

print(f"Data normalized to range [0, 1]")

# ## 5. Split Data

# Convert labels to categorical
y_categorical = to_categorical(y, num_classes=3)

# Split into train/validation/test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_categorical, test_size=0.15, random_state=42, stratify=y[:, 0]
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=42, stratify=y_temp[:, 0, :]
)

print(f"\nDataset split:")
print(f"Training: {len(X_train)} sequences")
print(f"Validation: {len(X_val)} sequences")
print(f"Test: {len(X_test)} sequences")

Data normalized to range [0, 1]

Dataset split:
Training: 5 sequences
Validation: 2 sequences
Test: 2 sequences


In [23]:
def create_lstm_model(input_shape, n_classes=3):
    """Create LSTM model for behavior classification"""
    model = Sequential([
        # Masking layer to handle padded values
        Masking(mask_value=0.0, input_shape=input_shape),

        # First LSTM layer
        LSTM(128, return_sequences=True, dropout=0.2),

        # Second LSTM layer
        LSTM(64, return_sequences=True, dropout=0.2),

        # Dense layers for each time step
        TimeDistributed(Dense(32, activation='relu')),
        Dropout(0.3),

        # Output layer
        TimeDistributed(Dense(n_classes, activation='softmax'))
    ])

    return model

In [24]:

# Create model
model = create_lstm_model(input_shape=(WINDOW_SIZE, 34))
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ## 7. Train the Model

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_lstm_behavior_model',  # Remove .h5 extension
        monitor='val_accuracy',
        save_best_only=True,
        save_format='tf',  # Use TensorFlow format instead of HDF5
        verbose=1
    )
]

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking_2 (Masking)         (None, 30, 34)            0         
                                                                 
 lstm_4 (LSTM)               (None, 30, 128)           83456     
                                                                 
 lstm_5 (LSTM)               (None, 30, 64)            49408     
                                                                 
 time_distributed_4 (TimeDis  (None, 30, 32)           2080      
 tributed)                                                       
                                                                 
 dropout_2 (Dropout)         (None, 30, 32)            0         
                                                                 
 time_distributed_5 (TimeDis  (None, 30, 3)            99        
 tributed)                                            

In [25]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# ## 8. Evaluate the Model

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Epoch 1/100
1/1 [==============================] - ETA: 0s - loss: 0.9799 - accuracy: 0.8733
Epoch 1: val_accuracy improved from -inf to 1.00000, saving model to best_lstm_behavior_model


INFO:tensorflow:Assets written to: best_lstm_behavior_model\assets


INFO:tensorflow:Assets written to: best_lstm_behavior_model\assets


1/1 [==============================] - 11s 11s/step - loss: 0.9799 - accuracy: 0.8733 - val_loss: 0.6621 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 2/100
1/1 [==============================] - ETA: 0s - loss: 0.7282 - accuracy: 0.9933
Epoch 2: val_accuracy did not improve from 1.00000
1/1 [==============================] - 0s 91ms/step - loss: 0.7282 - accuracy: 0.9933 - val_loss: 0.4612 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 3/100
1/1 [==============================] - ETA: 0s - loss: 0.5527 - accuracy: 1.0000
Epoch 3: val_accuracy did not improve from 1.00000
1/1 [==============================] - 0s 70ms/step - loss: 0.5527 - accuracy: 1.0000 - val_loss: 0.3216 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 4/100
1/1 [==============================] - ETA: 0s - loss: 0.3939 - accuracy: 1.0000
Epoch 4: val_accuracy did not improve from 1.00000
1/1 [==============================] - 0s 61ms/step - loss: 0.3939 - accuracy: 1.0000 - val_loss: 0.2277 - val_accuracy: 1.0000 - lr: 0.0010
Epoc

ValueError: object __array__ method not producing an array

<Figure size 1200x400 with 2 Axes>


Test Loss: 0.0049
Test Accuracy: 1.0000


In [27]:
# Get predictions
y_pred = model.predict(X_test)

# Convert to class labels (take majority vote across sequence)
y_test_labels = np.argmax(y_test, axis=2).flatten()
y_pred_labels = np.argmax(y_pred, axis=2).flatten()

# Classification report
behavior_names = ['Walking/Nothing', 'Suspicious', 'Running/Panic']
print("\nClassification Report:")
print(classification_report(y_test_labels, y_pred_labels,
                          target_names=behavior_names,
                          labels=[0, 1, 2],  # Add this line
                          digits=3))

# Confusion matrix
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=[0, 1, 2])  # Add labels here too
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=behavior_names,
            yticklabels=behavior_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# ## 10. Save the Model

1/1 [==============================] - 0s 20ms/step

Classification Report:
                 precision    recall  f1-score   support

Walking/Nothing      1.000     1.000     1.000        60
     Suspicious      0.000     0.000     0.000         0
  Running/Panic      0.000     0.000     0.000         0

      micro avg      1.000     1.000     1.000        60
      macro avg      0.333     0.333     0.333        60
   weighted avg      1.000     1.000     1.000        60



C:\Users\tomas\anaconda3\envs\tf210gpu\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\tomas\anaconda3\envs\tf210gpu\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\tomas\anaconda3\envs\tf210gpu\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\tomas\anacon

ValueError: object __array__ method not producing an array

ValueError: object __array__ method not producing an array

<Figure size 800x600 with 2 Axes>

In [26]:

# Save model
model.save('lstm_behavior_classifier.h5')
print("\nModel saved as 'lstm_behavior_classifier.h5'")

# Save normalization parameters
norm_params = {
    'global_min': float(global_min),
    'global_max': float(global_max),
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE
}

with open('normalization_params.json', 'w') as f:
    json.dump(norm_params, f)
print("Normalization parameters saved as 'normalization_params.json'")

1/1 [==============================] - 1s 1s/step

Classification Report:


ValueError: Number of classes, 1, does not match size of target_names, 3. Try specifying the labels parameter